# Phase 1.5 — MuSiQue 1b (chain-of-experts) driver

1b corpus pivot (2026-05-31). **Run All** 하면 됩니다. 순서:
1. **M3** — MuSiQue flat-1a baseline (corpus='musique', chain_steps=1). MC-acc>chance(0.25) + hop selectivity.
2. **M5** — 1b chain (L=3) + PASS bar: (a) hop-depth-selective lesion ∧ (b) S1 motif.
3. **M6** — control arm (logic_mc): hop-depth-selectivity 부재여야 정상(attribution).

단일 출처: `../../CONTEXT.md §Corpus`, ADR `../../docs/adr/0003`, memory `project_phase15_corpus_pivot_2026_05_31`, plan `hashed-orbiting-cerf.md`.

> **주의**: 첫 실행은 MuSiQue 22K rows를 e5-large-v2로 인코딩 → 시간 걸림(GPU 권장). 캐시 후 재실행은 빠름.

In [ ]:
import os, sys
from pathlib import Path

BASE = Path('/content/drive/MyDrive/sideproject')
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'BASE = {BASE} | device = {device}')
CACHE = str(BASE / 'out' / 'phase1_5' / 'cache')

## M3 — MuSiQue flat-1a baseline
weak/strong 여부보다 **MC-acc > 0.25** (학습되나) + hop selectivity 읽힘 확인이 목적. flat이 weak이어야 1b로 갈 이유가 성립.

In [ ]:
from phase1_5.data import MCCorpusConfig
from phase1_5.ablations import AblationRow
from phase1_5.engine_1a import run_engine_1a
from phase1_5.train import TrainConfig

cfg = MCCorpusConfig(corpus='musique', cache_root=CACHE,
                     max_train_samples=20000, max_val_samples=2000, max_test_samples=2000)
# dropout + best_metric='acc' (overfit control: MuSiQue memorises train at 0.0,
# and val CE explodes so 'acc' picks a usefully-trained ckpt, not epoch-0).
row = AblationRow(row_id='M3', name='MuSiQue flat-1a', k_routed=128,
                  lb_strategy='aux_free', corpus='musique', chain_steps=1, dropout=0.1)
res1a = run_engine_1a(ablation_row=row, corpus_cfg=cfg,
                      train_cfg=TrainConfig(epochs=15, lr=1e-3, k_target=4.0,
                                            best_metric='acc', seed=0),
                      device=device, seed=0, batch_size=64, progress=True,
                      out_dir=str(BASE / 'out' / 'phase1_5' / 'musique'))
h = res1a['history']
best_val = max(e.get('val_mc_acc', 0.0) for e in h)
print(f"train acc(last)={h[-1]['mc_acc']:.3f}  val acc(last)={h[-1].get('val_mc_acc'):.3f}  best val acc={best_val:.3f}")
print('hop gate verdict:', res1a['gate'].get('verdict'),
      '| adj_op:', round(res1a['operation_gate'].get('adj_operation', float('nan')), 3),
      '| raw-Q ceiling adj_op:',
      round(res1a['operation_ceiling_raw'].get('adj_operation', float('nan')), 3)
      if isinstance(res1a.get('operation_ceiling_raw'), dict) else res1a.get('operation_ceiling_raw'))
print('# 기대: val acc >> 0.25 (substrate 있음), flat hop selectivity는 weak/FAIL이어도 OK(→1b 동기)')

## M5 — 1b chain (L=3) + PASS bar
`passed = (a)hop-depth-selective lesion ∧ (b)motif op_purity>chance`. topk routing(K_active 고정, ADR 0002).

In [ ]:
from phase1_5.run_1b import run_1b_experiment

# Controls now: dropout=0.1, balance_train_hops=True (cap=5000/hop, MuSiQue만),
# best_metric='acc', routing='topk'. PASS(a) = re-specified "breadth grows with
# depth" (primary); 원래 last-step 기준도 투명하게 같이 출력.
out1b = run_1b_experiment(corpus_cfg=cfg,
                          train_cfg=TrainConfig(epochs=15, lr=1e-3, seed=0),
                          chain_steps=3, k_routed=128, k_active_target=4.0,
                          batch_size=64, device=device, seed=0)
h = out1b['train']['history']
best_val = max(e.get('val_mc_acc', 0.0) for e in h)
v = out1b['verdict']
print(f"1b train acc(last)={h[-1]['mc_acc']:.3f}  val acc(last)={h[-1].get('val_mc_acc'):.3f}  best val acc={best_val:.3f}")
print('PASSED:', v['passed'])
print('  (a) [primary] breadth grows w/ depth:', v['pass_a_breadth']['monotone'],
      '| breadth:', v['pass_a_breadth']['breadth'])
print('  (a) [original] last-step deep>shallow:', v['pass_a_laststep']['monotone_in_depth'],
      '| self-drop:', {k: round(x,3) for k,x in v['pass_a_laststep']['last_step_self_drop'].items()})
print('  (b) motif op_purity/chance:', round(v['s1_motif']['op_purity'],3), '/',
      round(v['s1_motif']['op_chance'],3), '->', v['pass_b_motif'])

## M6 — control arm (logic_mc)
logic-MC엔 hop 축이 없으므로 (a)causal이 **False여야 정상** → 'composition-substrate가 driver'라는 attribution 성립. (logic-MC는 hop 라벨 부재 → hops=[] → monotone False)

In [ ]:
cfg_logic = MCCorpusConfig(corpus='logic_mc', cache_root=CACHE,
                          max_train_samples=20000, max_val_samples=2000, max_test_samples=2000)
ctrl = run_1b_experiment(corpus_cfg=cfg_logic,
                         train_cfg=TrainConfig(epochs=15, lr=1e-3, seed=0),
                         chain_steps=3, k_routed=128, device=device, seed=0)
hc = ctrl['train']['history']
cv = ctrl['verdict']
print(f"control val acc(last)={hc[-1].get('val_mc_acc'):.3f}")
print('control (a)[primary] breadth grows:', cv['pass_a_breadth']['monotone'], '(False 기대)')
print('control (a)[original] last-step:', cv['pass_a_laststep']['monotone_in_depth'],
      '| hops present:', cv['pass_a_laststep']['hops'], '(logic-MC엔 hop축 없음 → [] 기대)')